In [ ]:
# pip install -e /root/capsule

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import glob
import pickle
import os
import logging
logging.basicConfig(
    level=logging.INFO, 
    format='%(filename)s:%(lineno)d - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
SESSION_KEYS = ["subject_id", "session_date"]

import json
import requests

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'
import seaborn as sns

from aind_behavior_gym.dynamic_foraging.task import CoupledBlockTask, UncoupledBlockTask
from aind_dynamic_foraging_models.generative_model import ForagerCollection

from aind_analysis_arch_result_access.han_pipeline import get_session_table, get_mle_model_fitting
from aind_analysis_arch_result_access.util.s3 import get_s3_pkl, get_s3_json

import aind_dynamic_foraging_population_analysis

In [ ]:
window_size = 10  # Analyze 10 trials before and after each switch


model_types = [
    'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax', 'ForagingCompareThreshold', 
    'CTTAvg'
    # 'CTTMeanReset'
]

task_types = [
    'Uncoupled Baiting', 'Uncoupled Without Baiting', 'Coupled Baiting', #'None'
]

later_stages = [
    'STAGE_FINAL', 'GRADUATED'
]

In [ ]:
# insert switch info

def extract_switches_and_run_lengths(choices: np.ndarray) -> [np.ndarray, np.ndarray]:
    """
    Extract switch trial indices and their associated run lengths.

    Parameters:
    -----------
    choices : np.ndarray
        Array of binary choices (0 or 1) for each trial
        
    Returns:
    --------
    switch_indices : np.ndarray
        Indices where switches occur (0-based)
    run_lengths : np.ndarray
        Run length preceding each switch
    """
    # Ensure choices is a numpy array
    choices = np.asarray(choices)

    # drop NaN values if present
    choices = choices[~np.isnan(choices)]
    
    if len(choices) < 2:
        raise ValueError("Choices array must have at least two elements to find switches.")
        return np.array([]), np.array([])
    
    
    # Find where consecutive choices differ
    switches = np.diff(choices) != 0
    switch_indices = np.where(switches)[0] + 1  # +1 because diff reduces length by 1
    
    # Calculate run lengths efficiently using vectorized operations
    if len(switch_indices) > 0:
        # Run lengths are differences between consecutive switch indices
        # First run length is from start (0) to first switch
        run_lengths = np.diff(np.concatenate(([0], switch_indices)))
    else:
        run_lengths = np.array([])
    
    # check if switch_indices and run_lengths are of the same length
    if len(switch_indices) != len(run_lengths):
        raise ValueError("Switch indices and run lengths should have the same length. " + 
                         f"Instead got {len(switch_indices)} and {len(run_lengths)}.")

    return switch_indices, run_lengths


# per session, get switch trial indices and run lengths
# insert switch trial indices and run lengths into df_session_history
def insert_switch_info(df_session_history):
    """Insert switch trial indices and run lengths into df_session_history."""
    switch_indices_list = []
    run_lengths_list = []
    
    for _, row in df_session_history.iterrows():
        choices = row['choice_history']
        if choices is not None:
            switch_indices, run_lengths = extract_switches_and_run_lengths(choices)
            switch_indices_list.append(switch_indices)
            run_lengths_list.append(run_lengths)
        else:
            switch_indices_list.append(np.array([]))
            run_lengths_list.append(np.array([]))
    
    df_session_history['switch_indices'] = switch_indices_list
    df_session_history['run_lengths'] = run_lengths_list
    
    return df_session_history

In [ ]:
# fetch fitted latent variables
URL = "https://api.allenneuraldynamics-test.org/v1/behavior_analysis/mle_fitting"
filter = {
    "nwb_name": "720956_2024-07-17_13-02-47.nwb",  # Session id,
    }
projection = {
    "analysis_results.fit_settings.agent_alias": 1,
    "_id": 0,
}
response = requests.get(URL, params={"filter": json.dumps(filter), "projection": json.dumps(projection)})
fitted_models = [item["analysis_results"]["fit_settings"]["agent_alias"] for item in response.json()]
fitted_models


In [ ]:
filter = {
    "nwb_name": "720956_2024-07-17_13-02-47.nwb",  # Session id,
    "analysis_results.fit_settings.agent_alias": "QLearning_L1F1_CK1_softmax",  # Bari2019 model
}
projection = {
    "analysis_results.params": 1,
    "analysis_results.fitted_latent_variables": 1,
    "_id": 0,
}
response = requests.get(URL, params={"filter": json.dumps(filter), "projection": json.dumps(projection)})
record_dict = response.json()[0]

# Fitted parameters
params = record_dict["analysis_results"]["params"]

# Fitted latent variables
fitted_latent = record_dict["analysis_results"]["fitted_latent_variables"]

print(params)
print(fitted_latent.keys())
print(np.array(fitted_latent["q_value"]).shape)

In [ ]:
df_session_history[df_session_history['nwb_name']=="720956_2024-07-17_13-02-47.nwb"].iloc[0]

## load data

In [ ]:
# load df_session_history

df_session_history = pd.read_pickle(
    os.path.expanduser("~/capsule/analysis_result/df_session_history_250714.pkl")
)

print(f"Loaded {len(df_session_history)} rows of session history data.")


# insert switch info into df_session_history
df_session_history = insert_switch_info(df_session_history)
print(f"Inserted switch info into df_session_history. Now has {len(df_session_history)} rows.")

df_session_history.head()

In [ ]:
df_session_history.subject_id.value_counts()

## load model fitting results 

In [ ]:
# # if already saved, load the df
# df_model_fitting_relevant = pd.read_pickle(
#     os.path.expanduser("~/capsule/analysis_result/df_model_fitting_relevant_250714.pkl")
# )

# print(f"Loaded {len(df_model_fitting_relevant)} rows of relevant model fitting data.")

In [ ]:
# df_model_fitting_relevant.head()

## load simulated data

In [ ]:
# load simulation results: hattori, bari, foraging
simulation_saved_file = os.path.expanduser("~/capsule/analysis_result/dict_simulation_results_250716.pkl")
with open(simulation_saved_file, 'rb') as f:
    dict_simulation_results = pickle.load(f)

In [ ]:
df_sim_tmp = dict_simulation_results[(model_types[0], task_types[0])]
print(f"Loaded {len(df_sim_tmp)} rows of simulation results for model {model_types[0]} and task {task_types[0]}.")

df_sim_tmp.head()

df_sim_tmp.subject_id.value_counts()

In [ ]:
# load simulation results: ctt_avg
with open("/root/capsule/analysis_result/dict_simulation_results-CTTAvg-251014.pkl", 'rb') as f:
    dict_simulation_results_ctt_avg = pickle.load(f)

In [ ]:
# # load and merge simulation results: CTTAvg, archived
# with open('/root/capsule/analysis_result/dict_simulation_results-CTTAvg-251007-coupled_baiting.pkl', 'rb') as fi:
#     dict_simulation_results_ctt_avg_coupled_baiting = pickle.load(fi)

# with open('/root/capsule/analysis_result/dict_simulation_results-CTTAvg-251007-uncoupled_baiting.pkl', 'rb') as fi:
#     dict_simulation_results_ctt_avg_uncoupled_baiting = pickle.load(fi)

# # # empty
# # with open('/root/capsule/analysis_result/dict_simulation_results-CTTAvg-251007-uncoupled_no_baiting-0_322.pkl', 'rb') as fi:
# #     dict_simulation_results_ctt_avg_uncoupled_no_baiting_0_322 = pickle.load(fi)

# with open('/root/capsule/analysis_result/dict_simulation_results-CTTAvg-251007-uncoupled_no_baiting-322_end.pkl', 'rb') as fi:
#     dict_simulation_results_ctt_avg_uncoupled_no_baiting_322_end = pickle.load(fi)


# # merge all CTTAvg simulation results
# dict_simulation_results_ctt_avg_all = {
#     **dict_simulation_results_ctt_avg_coupled_baiting,
#     **dict_simulation_results_ctt_avg_uncoupled_baiting,
#     **dict_simulation_results_ctt_avg_uncoupled_no_baiting_322_end
# }

# dict_simulation_results_ctt_avg_all.keys()

In [ ]:
# load simulation results: ctt_mean_reset
with open("/root/capsule/analysis_result/dict_simulation_results-CTTMeanReset-251014.pkl", 'rb') as f:
    dict_simulation_results_ctt_mean_reset = pickle.load(f)

In [ ]:
# merge into dict_simulation_results
dict_simulation_results = {
    **dict_simulation_results,
    **dict_simulation_results_ctt_avg,
    # **dict_simulation_results_ctt_mean_reset
}

# check keys
dict_simulation_results.keys()

# Global level analysis

In [ ]:
# Compute switch probabilities around switch events

def compute_switch_probabilities_around_switches(
    df_session_history, 
    window_size=10
):
    """
    Compute switch events for trials surrounding switch events.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        DataFrame with session data including 'choice_history' and 'switch_indices'
    window_size : int
        Number of trials before and after each switch to analyze
        
    Returns:
    --------
    switch_analysis_data : list of dicts
        Each dict contains switch event data for pooling across sessions
    """
    switch_analysis_data = []
    
    for session_idx, row in df_session_history.iterrows():
        choices = row['choice_history']
        switch_indices = row['switch_indices']
        
        if choices is None or len(switch_indices) == 0:
            continue
            
        choices = np.asarray(choices)
        # Drop NaN values from choices
        choices = choices[~np.isnan(choices)]
        
    
        # For each switch in this session
        for switch_idx in switch_indices:
            # Skip if switch_idx is 0 (can't analyze switch at first trial)
            if switch_idx == 0:
                continue
                
            # Define window around switch
            start_idx = max(0, switch_idx - window_size)
            end_idx = min(len(choices), switch_idx + window_size + 1)
            
            # Initialize array for switch events (0/1, not probabilities yet)
            switch_events = np.full(2 * window_size + 1, np.nan)
            
            # Calculate switch events for each position in the window
            for trial_pos in range(start_idx, end_idx):
                # Skip if this is the first trial (no previous trial to compare)
                if trial_pos == 0:
                    continue
                    
                # Calculate switch indicator: did choice change from previous trial?
                switch_indicator = choices[trial_pos] != choices[trial_pos - 1]
                
                # Relative position to the switch trial
                rel_pos = trial_pos - switch_idx
                
                # Index in switch_events array
                array_idx = rel_pos + window_size
                
                # Only store if within our desired window
                if 0 <= array_idx < len(switch_events):
                    switch_events[array_idx] = float(switch_indicator)  # 0.0 or 1.0
            
            # Store switch event data
            switch_event_data = {
                'session_idx': session_idx,
                'subject_id': row.get('subject_id', None),
                'session_date': row.get('session_date', None),
                'switch_trial': switch_idx,
                'switch_events': switch_events,
                'relative_positions': np.arange(-window_size, window_size + 1),
                'window_size': window_size
            }
            switch_analysis_data.append(switch_event_data)
    
    return switch_analysis_data


def create_pooled_switch_analysis(switch_analysis_data):
    """
    Pool switch analysis data across sessions to compute mean probabilities.
    
    Parameters:
    -----------
    switch_analysis_data : list of dicts
        Output from compute_switch_probabilities_around_switches
        
    Returns:
    --------
    df_pooled : pd.DataFrame
        DataFrame with pooled switch probabilities by relative position
    """
    if not switch_analysis_data:
        return pd.DataFrame()
    
    # Stack all switch event arrays
    all_switch_events = np.vstack([event['switch_events'] for event in switch_analysis_data])  # Updated
    relative_positions = switch_analysis_data[0]['relative_positions']
    
    # Compute statistics across all switch events to get probabilities
    pooled_data = []
    for i, rel_pos in enumerate(relative_positions):
        valid_data = all_switch_events[:, i][~np.isnan(all_switch_events[:, i])]
        
        if len(valid_data) > 0:
            pooled_data.append({
                'relative_position': rel_pos,
                'switch_probability': np.mean(valid_data),  # NOW this is actually a probability
                'switch_probability_sem': np.std(valid_data) / np.sqrt(len(valid_data)),
                'n_events': len(valid_data),
                'switch_probability_std': np.std(valid_data)
            })
    
    return pd.DataFrame(pooled_data)


def plot_switch_probability_around_switches(df_pooled_switches):
    """Plot switch probability as a function of relative trial position."""
    plt.figure(figsize=(10, 6))
    
    x = df_pooled_switches['relative_position']
    y = df_pooled_switches['switch_probability']
    yerr = df_pooled_switches['switch_probability_std']
    
    plt.errorbar(x, y, yerr=yerr, marker='o', capsize=3)
    plt.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='Switch trial')
    plt.xlabel('Relative trial position')
    plt.ylabel('Switch probability')
    plt.title('Switch Probability Around Switch Events')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Analyze switch probabilities conditioned on reward outcome

def compute_conditional_switch_probabilities(df_session_history, window_size=10):
    """
    Compute switch probabilities around switch events, conditioned on reward outcome.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        DataFrame with session data including 'choice_history', 'switch_indices', and 'reward_history'
    window_size : int
        Number of trials before and after each switch to analyze
        
    Returns:
    --------
    conditional_analysis_data : dict
        Contains 'rewarded' and 'unrewarded' keys, each with list of switch event data
    """
    conditional_analysis_data = {'rewarded': [], 'unrewarded': []}
    
    for session_idx, row in df_session_history.iterrows():
        choices = row['choice_history']
        switch_indices = row['switch_indices']
        rewards = row.get('reward_history', None)  # Assuming reward_history exists
        
        if choices is None or len(switch_indices) == 0 or rewards is None:
            continue
            
        choices = np.asarray(choices)
        rewards = np.asarray(rewards)
        
        # Drop NaN values from choices and corresponding rewards
        valid_mask = ~np.isnan(choices)
        choices = choices[valid_mask]
        rewards = rewards[valid_mask]
    
        # For each switch in this session
        for switch_idx in switch_indices:
            # Skip if switch_idx is 0 or out of bounds
            if switch_idx == 0 or switch_idx >= len(rewards):
                continue
                
            # Check if switch trial was rewarded
            switch_rewarded = rewards[switch_idx] > 0
            condition = 'rewarded' if switch_rewarded else 'unrewarded'
                
            # Define window around switch
            start_idx = max(0, switch_idx - window_size)
            end_idx = min(len(choices), switch_idx + window_size + 1)
            
            # Initialize array for switch events
            switch_events = np.full(2 * window_size + 1, np.nan)
            
            # Calculate switch events for each position in the window
            for trial_pos in range(start_idx, end_idx):
                if trial_pos == 0:
                    continue
                    
                switch_indicator = choices[trial_pos] != choices[trial_pos - 1]
                rel_pos = trial_pos - switch_idx
                array_idx = rel_pos + window_size
                
                if 0 <= array_idx < len(switch_events):
                    switch_events[array_idx] = float(switch_indicator)
            
            # Store switch event data in appropriate condition
            switch_event_data = {
                'session_idx': session_idx,
                'subject_id': row.get('subject_id', None),
                'session_date': row.get('session_date', None),
                'switch_trial': switch_idx,
                'switch_rewarded': switch_rewarded,
                'switch_events': switch_events,
                'relative_positions': np.arange(-window_size, window_size + 1),
                'window_size': window_size
            }
            conditional_analysis_data[condition].append(switch_event_data)
    
    return conditional_analysis_data


def analyze_post_switch_probability_by_reward(conditional_analysis_data):
    """
    Analyze switch probability on trial immediately after switch, by reward condition.
    
    Parameters:
    -----------
    conditional_analysis_data : dict
        Output from compute_conditional_switch_probabilities
        
    Returns:
    --------
    results : dict
        Contains switch probabilities for trial +1 by reward condition
    """
    results = {}
    
    for condition in ['rewarded', 'unrewarded']:
        events = conditional_analysis_data[condition]
        if not events:
            results[condition] = {'probability': np.nan, 'sem': np.nan, 'n': 0}
            continue
            
        # Extract switch events for trial +1 (index window_size + 1)
        window_size = events[0]['window_size']
        post_switch_idx = window_size + 1  # +1 trial after switch
        
        post_switch_events = []
        for event in events:
            if post_switch_idx < len(event['switch_events']):
                value = event['switch_events'][post_switch_idx]
                if not np.isnan(value):
                    post_switch_events.append(value)
        
        if post_switch_events:
            prob = np.mean(post_switch_events)
            sem = np.std(post_switch_events) / np.sqrt(len(post_switch_events))
            results[condition] = {
                'probability': prob, 
                'sem': sem, 
                'n': len(post_switch_events)
            }
        else:
            results[condition] = {'probability': np.nan, 'sem': np.nan, 'n': 0}
    
    return results


def plot_conditional_switch_analysis(conditional_analysis_data):
    """Plot switch probabilities by reward condition."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    colors = {'rewarded': 'green', 'unrewarded': 'red'}
    
    # Plot full time course
    for condition in ['rewarded', 'unrewarded']:
        df_pooled = create_pooled_switch_analysis(conditional_analysis_data[condition])
        if not df_pooled.empty:
            ax1.errorbar(df_pooled['relative_position'], 
                        df_pooled['switch_probability'],
                        yerr=df_pooled['switch_probability_sem'],
                        marker='o', label=f'{condition.capitalize()}',
                        color=colors[condition], capsize=3)
    
    ax1.axvline(x=0, color='black', linestyle='--', alpha=0.7, label='Switch trial')
    ax1.set_xlabel('Relative trial position')
    ax1.set_ylabel('Switch probability')
    ax1.set_title('Switch Probability by Reward Condition')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot just the post-switch probability
    results = analyze_post_switch_probability_by_reward(conditional_analysis_data)
    
    conditions = ['rewarded', 'unrewarded']
    probs = [results[c]['probability'] for c in conditions]
    sems = [results[c]['sem'] for c in conditions]
    ns = [results[c]['n'] for c in conditions]
    
    bars = ax2.bar(conditions, probs, yerr=sems, capsize=5, 
                   color=[colors[c] for c in conditions], alpha=0.7)
    
    # Add sample size annotations
    for i, (bar, n) in enumerate(zip(bars, ns)):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + sems[i] + 0.01,
                f'n={n}', ha='center', va='bottom')
    
    ax2.set_ylabel('Switch probability')
    ax2.set_title('Switch Probability on Trial +1')
    ax2.set_ylim(0, None)
    
    plt.tight_layout()
    plt.show()
    
    return results

In [ ]:
def compute_conditional_switch_probabilities(df_session_history, window_size=10):
    """
    Compute switch probabilities around switch events, conditioned on reward outcome and run length.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        DataFrame with session data including 'choice_history', 'switch_indices', 'reward_history', and 'run_lengths'
    window_size : int
        Number of trials before and after each switch to analyze
        
    Returns:
    --------
    conditional_analysis_data : dict
        Contains nested keys: ['rewarded']['run_length_1'], ['rewarded']['run_length_gt1'], 
        ['unrewarded']['run_length_1'], ['unrewarded']['run_length_gt1']
        Each with list of switch event data
    """
    conditional_analysis_data = {
        'rewarded': {'run_length_1': [], 'run_length_gt1': []},
        'unrewarded': {'run_length_1': [], 'run_length_gt1': []}
    }
    
    for session_idx, row in df_session_history.iterrows():
        choices = row['choice_history']
        switch_indices = row['switch_indices']
        run_lengths = row['run_lengths']
        rewards = row.get('reward_history', None)
        
        if choices is None or len(switch_indices) == 0 or rewards is None or len(run_lengths) == 0:
            continue
            
        choices = np.asarray(choices)
        rewards = np.asarray(rewards)
        run_lengths = np.asarray(run_lengths)
        
        # Drop NaN values from choices and corresponding rewards
        valid_mask = ~np.isnan(choices)
        choices = choices[valid_mask]
        rewards = rewards[valid_mask]
    
        # For each switch in this session
        for switch_idx_pos, switch_idx in enumerate(switch_indices):
            # Skip if switch_idx is 0 or out of bounds
            if switch_idx == 0 or switch_idx >= len(rewards) or switch_idx_pos >= len(run_lengths):
                continue
                
            # Check if switch trial was rewarded
            switch_rewarded = rewards[switch_idx] > 0
            reward_condition = 'rewarded' if switch_rewarded else 'unrewarded'
            
            # Get run length preceding this switch
            preceding_run_length = run_lengths[switch_idx_pos]
            run_length_condition = 'run_length_1' if preceding_run_length == 1 else 'run_length_gt1'
                
            # Define window around switch
            start_idx = max(0, switch_idx - window_size)
            end_idx = min(len(choices), switch_idx + window_size + 1)
            
            # Initialize array for switch events
            switch_events = np.full(2 * window_size + 1, np.nan)
            
            # Calculate switch events for each position in the window
            for trial_pos in range(start_idx, end_idx):
                if trial_pos == 0:
                    continue
                    
                switch_indicator = choices[trial_pos] != choices[trial_pos - 1]
                rel_pos = trial_pos - switch_idx
                array_idx = rel_pos + window_size
                
                if 0 <= array_idx < len(switch_events):
                    switch_events[array_idx] = float(switch_indicator)
            
            # Store switch event data in appropriate condition
            switch_event_data = {
                'session_idx': session_idx,
                'subject_id': row.get('subject_id', None),
                'session_date': row.get('session_date', None),
                'switch_trial': switch_idx,
                'switch_rewarded': switch_rewarded,
                'preceding_run_length': preceding_run_length,
                'switch_events': switch_events,
                'relative_positions': np.arange(-window_size, window_size + 1),
                'window_size': window_size
            }
            conditional_analysis_data[reward_condition][run_length_condition].append(switch_event_data)
    
    return conditional_analysis_data


def analyze_post_switch_probability_by_reward_and_run_length(conditional_analysis_data):
    """
    Analyze switch probability on trial immediately after switch, by reward and run length conditions.
    
    Parameters:
    -----------
    conditional_analysis_data : dict
        Output from compute_conditional_switch_probabilities (modified version)
        
    Returns:
    --------
    results : dict
        Contains switch probabilities for trial +1 by reward and run length conditions
    """
    results = {}
    
    for reward_condition in ['rewarded', 'unrewarded']:
        results[reward_condition] = {}
        
        for run_condition in ['run_length_1', 'run_length_gt1']:
            events = conditional_analysis_data[reward_condition][run_condition]
            
            if not events:
                results[reward_condition][run_condition] = {
                    'probability': np.nan, 'sem': np.nan, 'n': 0
                }
                continue
                
            # Extract switch events for trial +1 (index window_size + 1)
            window_size = events[0]['window_size']
            post_switch_idx = window_size + 1  # +1 trial after switch
            
            post_switch_events = []
            for event in events:
                if post_switch_idx < len(event['switch_events']):
                    value = event['switch_events'][post_switch_idx]
                    if not np.isnan(value):
                        post_switch_events.append(value)
            
            if post_switch_events:
                prob = np.mean(post_switch_events)
                sem = np.std(post_switch_events) / np.sqrt(len(post_switch_events))
                results[reward_condition][run_condition] = {
                    'probability': prob, 
                    'sem': sem, 
                    'n': len(post_switch_events)
                }
            else:
                results[reward_condition][run_condition] = {
                    'probability': np.nan, 'sem': np.nan, 'n': 0
                }
    
    return results


def plot_conditional_switch_analysis_with_run_length(conditional_analysis_data):
    """Plot switch probabilities by reward condition and run length."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    colors = {
        'run_length_1': 'blue',
        'run_length_gt1': 'orange'
    }
    
    reward_conditions = ['rewarded', 'unrewarded']
    
    # Plot full time course for each reward condition
    for reward_idx, reward_condition in enumerate(reward_conditions):
        ax = axes[reward_idx, 0]
        
        for run_condition in ['run_length_1', 'run_length_gt1']:
            df_pooled = create_pooled_switch_analysis(
                conditional_analysis_data[reward_condition][run_condition]
            )
            
            if not df_pooled.empty:
                label = 'Run length = 1' if run_condition == 'run_length_1' else 'Run length > 1'
                ax.errorbar(df_pooled['relative_position'], 
                           df_pooled['switch_probability'],
                           yerr=df_pooled['switch_probability_sem'],
                           marker='o', label=label,
                           color=colors[run_condition], capsize=3)
        
        ax.axvline(x=0, color='black', linestyle='--', alpha=0.7, label='Switch trial')
        ax.set_xlabel('Relative trial position')
        ax.set_ylabel('Switch probability')
        ax.set_title(f'Switch Probability - {reward_condition.capitalize()} Switches')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # Plot post-switch probabilities (grouped bar chart)
    results = analyze_post_switch_probability_by_reward_and_run_length(conditional_analysis_data)
    
    # Prepare data for grouped bar chart
    x_labels = ['Rewarded\nRun=1', 'Rewarded\nRun>1', 'Unrewarded\nRun=1', 'Unrewarded\nRun>1']
    probabilities = []
    sems = []
    ns = []
    bar_colors = []
    
    for reward_condition in ['rewarded', 'unrewarded']:
        for run_condition in ['run_length_1', 'run_length_gt1']:
            data = results[reward_condition][run_condition]
            probabilities.append(data['probability'])
            sems.append(data['sem'])
            ns.append(data['n'])
            bar_colors.append(colors[run_condition])
    
    ax = axes[0, 1]
    x_pos = np.arange(len(x_labels))
    bars = ax.bar(x_pos, probabilities, yerr=sems, capsize=5, 
                  color=bar_colors, alpha=0.7)
    
    # Add sample size annotations
    for i, (bar, n) in enumerate(zip(bars, ns)):
        height = bar.get_height()
        if not np.isnan(height):
            ax.text(bar.get_x() + bar.get_width()/2., height + sems[i] + 0.01,
                    f'n={n}', ha='center', va='bottom', fontsize=10)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels)
    ax.set_ylabel('Switch probability at t+1')
    ax.set_title('Switch Probability at t+1 by Condition')
    ax.set_ylim(0, None)
    ax.grid(True, alpha=0.3)
    
    # Create a summary table in the bottom right subplot
    ax = axes[1, 1]
    ax.axis('off')  # Turn off axis
    
    # Create table data
    table_data = []
    headers = ['Condition', 'Probability', 'SEM', 'N']
    
    for reward_condition in ['rewarded', 'unrewarded']:
        for run_condition in ['run_length_1', 'run_length_gt1']:
            data = results[reward_condition][run_condition]
            run_label = 'Run=1' if run_condition == 'run_length_1' else 'Run>1'
            condition_label = f"{reward_condition.capitalize()}, {run_label}"
            
            prob_str = f"{data['probability']:.3f}" if not np.isnan(data['probability']) else 'N/A'
            sem_str = f"{data['sem']:.3f}" if not np.isnan(data['sem']) else 'N/A'
            
            table_data.append([condition_label, prob_str, sem_str, str(data['n'])])
    
    # Create table
    table = ax.table(cellText=table_data, colLabels=headers, 
                     cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    
    ax.set_title('Summary Statistics', pad=20)
    
    plt.suptitle('Switch Probability Analysis by Reward Outcome and Run Length', 
                 fontsize=16, y=0.98)
    plt.tight_layout()
    plt.show()
    
    return results

In [ ]:
# Compute switch probabilities around switches for all sessions
switch_analysis_data_task_type = compute_switch_probabilities_around_switches(df_session_history, window_size)
print(f"Found {len(switch_analysis_data_task_type)} switch events across all sessions")

# Create pooled analysis
df_pooled_switches_task_type = create_pooled_switch_analysis(switch_analysis_data_task_type)
print("\nPooled switch probabilities around switch events:")
print(df_pooled_switches_task_type)

plot_switch_probability_around_switches(df_pooled_switches_task_type)


# Compute conditional switch probabilities for the current task type
conditional_analysis_data_task_type = compute_conditional_switch_probabilities(df_session_history=df_session_history, window_size=window_size)

print(f"Rewarded switches: {len(conditional_analysis_data_task_type['rewarded'])}")
print(f"Unrewarded switches: {len(conditional_analysis_data_task_type['unrewarded'])}")

# Analyze and plot
# results_task_type = plot_conditional_switch_analysis(conditional_analysis_data_task_type)
results_task_type = plot_conditional_switch_analysis_with_run_length(conditional_analysis_data_task_type)


# Print numerical results
print("\nSwitch probability on trial immediately after switch:")
for rew_condition, data in results_task_type.items():
    for run_condition, run_data in data.items():
        if run_data['n'] > 0:
            print(f"{rew_condition.capitalize()} - {run_condition}: {run_data['probability']:.3f} ± {run_data['sem']:.3f} (n={run_data['n']})")

In [ ]:
# Analyze switch probabilities for each task type

for task_type in task_types:
    print(f"Analyzing p_switch for {task_type}...")

    # Filter sessions for the current task type and later stages
    df_session_history_task_type = df_session_history[
        (df_session_history['curriculum_name'] == task_type) &
        (df_session_history['current_stage_actual'].isin(later_stages))]

    
    # Compute switch probabilities around switches for this task type
    switch_analysis_data_task_type = compute_switch_probabilities_around_switches(df_session_history_task_type, window_size)
    print(f"Found {len(switch_analysis_data_task_type)} switch events across all sessions")

    # Create pooled analysis
    df_pooled_switches_task_type = create_pooled_switch_analysis(switch_analysis_data_task_type)
    print("\nPooled switch probabilities around switch events:")
    print(df_pooled_switches_task_type)

    plot_switch_probability_around_switches(df_pooled_switches_task_type)


    # Compute conditional switch probabilities for the current task type
    print(f"Computing conditional switch probabilities for {task_type}...")
    conditional_analysis_data_task_type = compute_conditional_switch_probabilities(df_session_history_task_type, window_size)

    print(f"Rewarded switches: {len(conditional_analysis_data_task_type['rewarded'])}")
    print(f"Unrewarded switches: {len(conditional_analysis_data_task_type['unrewarded'])}")

    # Analyze and plot
    # results_task_type = plot_conditional_switch_analysis(conditional_analysis_data_task_type)
    results_task_type = plot_conditional_switch_analysis_with_run_length(conditional_analysis_data_task_type)
    

    # Print numerical results
    print("\nSwitch probability on trial immediately after switch:")
    for rew_condition, data in results_task_type.items():
        for run_condition, run_data in data.items():
            if run_data['n'] > 0:
                print(f"{rew_condition.capitalize()} - {run_condition}: {run_data['probability']:.3f} ± {run_data['sem']:.3f} (n={run_data['n']})")

In [ ]:
# choose sessions of a specific subject

subject_id = "724584"  # Replace with the desired subject ID
df_session_history_subject = df_session_history[df_session_history['subject_id'] == subject_id]


# Compute switch probabilities around all switch events
switch_analysis_data = compute_switch_probabilities_around_switches(df_session_history_subject, window_size)
print(f"Found {len(switch_analysis_data)} switch events across all sessions")

# Create pooled analysis
df_pooled_switches = create_pooled_switch_analysis(switch_analysis_data)

# Display results
print("\nPooled switch probabilities around switch events:")
print(df_pooled_switches)

plot_switch_probability_around_switches(df_pooled_switches)

In [ ]:
window_size = 10  # Analyze 10 trials before and after each switch

# Compute conditional switch probabilities
conditional_analysis_data = compute_conditional_switch_probabilities(df_session_history, window_size)

print(f"Rewarded switches: {len(conditional_analysis_data['rewarded'])}")
print(f"Unrewarded switches: {len(conditional_analysis_data['unrewarded'])}")

# Analyze and plot
results = plot_conditional_switch_analysis_with_run_length(conditional_analysis_data)

# Print numerical results
print("\nSwitch probability on trial immediately after switch:")
for rew_condition, data in results_task_type.items():
    for run_condition, run_data in data.items():
        if run_data['n'] > 0:
            print(f"{rew_condition.capitalize()} - {run_condition}: {run_data['probability']:.3f} ± {run_data['sem']:.3f} (n={run_data['n']})")

## compare with simulated data

In [ ]:
# Analyze switch probabilities for each model type and task type

for model_type in model_types:
    for task_type in task_types:
        key = (model_type, task_type)
        print(f"Simulation results for {key}...")
        df_simulation_results = dict_simulation_results[key]

        # get switch trial indices and run lengths
        df_simulation_results = insert_switch_info(df_simulation_results)


        # Compute switch probabilities around all switch events
        # switch_analysis_data_simulation = compute_switch_probabilities_around_switches(df_simulation_results, window_size)

        # print(f"Found {len(switch_analysis_data_simulation)} switch events across all sessions")

        # # Create pooled analysis
        # df_pooled_switches_simulation = create_pooled_switch_analysis(switch_analysis_data_simulation)

        # # Display results
        # print("\nPooled switch probabilities around switch events:")
        # print(df_pooled_switches_simulation)

        # # Plot the switch probabilities
        # plot_switch_probability_around_switches(df_pooled_switches_simulation)


        # Compute conditional switch probabilities
        # Compute conditional switch probabilities
        conditional_analysis_data = compute_conditional_switch_probabilities(df_simulation_results, window_size)

        print(f"Rewarded switches: {len(conditional_analysis_data['rewarded'])}")
        print(f"Unrewarded switches: {len(conditional_analysis_data['unrewarded'])}")

        # Analyze and plot
        results = plot_conditional_switch_analysis_with_run_length(conditional_analysis_data)

        # Print numerical results
        print("\nSwitch probability on trial immediately after switch:")
        for rew_condition, data in results.items():
            for run_condition, run_data in data.items():
                if run_data['n'] > 0:
                    print(f"{rew_condition.capitalize()} - {run_condition}: {run_data['probability']:.3f} ± {run_data['sem']:.3f} (n={run_data['n']})")

## comparison plot

In [ ]:
# without further conditioning on run length

# def plot_pswitch_comparison(df_session_history, dict_simulation_results, window_size=10):
#     """
#     Plot comparison of switch probabilities at t+1 for data vs model results.
    
#     Parameters:
#     -----------
#     df_session_history : pd.DataFrame
#         Real behavioral data
#     dict_simulation_results : dict
#         Dictionary containing simulation results for different models and task types
#     window_size : int
#         Window size for switch analysis
#     """
#     model_types = ['ForagingCompareThreshold', 'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax']
#     task_types = ['Uncoupled Baiting', 'Uncoupled Without Baiting', 'Coupled Baiting']
#     later_stages = ['STAGE_FINAL', 'GRADUATED']
    
#     # Create figure with subplots
#     fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
#     # Colors for different data types
#     colors = {
#         'Data': '#2E86AB',
#         'ForagingCompareThreshold': '#A23B72',
#         'QLearning_L2F1_softmax': '#F18F01',
#         'QLearning_L1F1_CK1_softmax': '#C73E1D'
#     }
    
#     # Bar width and positions
#     bar_width = 0.15
#     x_positions = np.arange(2)  # rewarded, unrewarded
    
#     for task_idx, task_type in enumerate(task_types):
#         ax = axes[task_idx]
        
#         # Collect all results for this task type
#         all_results = {}
        
#         # Process real data
#         print(f"Processing data for {task_type}...")
#         df_task = df_session_history[
#             (df_session_history['curriculum_name'] == task_type) &
#             (df_session_history['current_stage_actual'].isin(later_stages))
#         ]
        
#         if len(df_task) > 0:
#             conditional_data = compute_conditional_switch_probabilities(df_task, window_size)
#             data_results = analyze_post_switch_probability_by_reward(conditional_data)
#             all_results['Data'] = data_results
        
#         # Process model results
#         for model_type in model_types:
#             key = (model_type, task_type)
#             if key in dict_simulation_results:
#                 print(f"Processing {model_type} for {task_type}...")
#                 df_sim = dict_simulation_results[key].copy()
#                 df_sim = insert_switch_info(df_sim)
#                 conditional_sim = compute_conditional_switch_probabilities(df_sim, window_size)
#                 sim_results = analyze_post_switch_probability_by_reward(conditional_sim)
#                 all_results[model_type] = sim_results
        
#         # Plot bars for each condition (rewarded/unrewarded)
#         for cond_idx, condition in enumerate(['rewarded', 'unrewarded']):
#             x_pos = x_positions[cond_idx]
            
#             for data_idx, (data_type, results) in enumerate(all_results.items()):
#                 if condition in results:
#                     prob = results[condition]['probability']
#                     sem = results[condition]['sem']
#                     n = results[condition]['n']
                    
#                     # Position bars side by side
#                     bar_pos = x_pos + (data_idx - 1.5) * bar_width
                    
#                     # Plot bar
#                     bar = ax.bar(bar_pos, prob, bar_width, 
#                                yerr=sem, capsize=3,
#                                color=colors[data_type], alpha=0.7,
#                                label=data_type if cond_idx == 0 else "")
                    
#                     # Add sample size annotation
#                     if not np.isnan(prob):
#                         ax.text(bar_pos, prob + sem + 0.01, f'n={n}',
#                                ha='center', va='bottom', fontsize=8, rotation=90)
        
#         # Customize subplot
#         ax.set_xticks(x_positions)
#         ax.set_xticklabels(['Rewarded', 'Unrewarded'])
#         ax.set_ylabel('Switch probability at t+1')
#         ax.set_title(f'{task_type}')
#         ax.set_ylim(0, None)
#         ax.grid(True, alpha=0.3)
        
#         # Add legend only to first subplot
#         if task_idx == 0:
#             ax.legend(bbox_to_anchor=(0, 1), loc='upper left')
    
#     plt.suptitle('Switch Probability at t+1: Data vs Models by Task Type and Reward Condition', 
#                  fontsize=14, y=1.02)
#     plt.tight_layout()
#     plt.show()
    
#     # Print summary statistics
#     print("\n" + "="*80)
#     print("SUMMARY STATISTICS")
#     print("="*80)
    
#     for task_type in task_types:
#         print(f"\n{task_type}:")
#         print("-" * len(task_type))
        
#         # Real data
#         df_task = df_session_history[
#             (df_session_history['curriculum_name'] == task_type) &
#             (df_session_history['current_stage_actual'].isin(later_stages))
#         ]
        
#         if len(df_task) > 0:
#             conditional_data = compute_conditional_switch_probabilities(df_task, window_size)
#             data_results = analyze_post_switch_probability_by_reward(conditional_data)
            
#             print("Data:")
#             for condition in ['rewarded', 'unrewarded']:
#                 if condition in data_results:
#                     prob = data_results[condition]['probability']
#                     sem = data_results[condition]['sem']
#                     n = data_results[condition]['n']
#                     print(f"  {condition}: {prob:.3f} ± {sem:.3f} (n={n})")
        
#         # Model results
#         for model_type in model_types:
#             key = (model_type, task_type)
#             if key in dict_simulation_results:
#                 df_sim = dict_simulation_results[key].copy()
#                 df_sim = insert_switch_info(df_sim)
#                 conditional_sim = compute_conditional_switch_probabilities(df_sim, window_size)
#                 sim_results = analyze_post_switch_probability_by_reward(conditional_sim)
                
#                 print(f"{model_type}:")
#                 for condition in ['rewarded', 'unrewarded']:
#                     if condition in sim_results:
#                         prob = sim_results[condition]['probability']
#                         sem = sim_results[condition]['sem']
#                         n = sim_results[condition]['n']
#                         print(f"  {condition}: {prob:.3f} ± {sem:.3f} (n={n})")


In [ ]:
def plot_pswitch_comparison(df_session_history, dict_simulation_results, window_size=10):
    """
    Plot comparison of switch probabilities at t+1 for data vs model results, 
    conditioned on reward outcome and run length.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        Real behavioral data
    dict_simulation_results : dict
        Dictionary containing simulation results for different models and task types
    window_size : int
        Window size for switch analysis
    """
    model_types = [
        'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax', 'ForagingCompareThreshold', 
        'CTTAvg'
        # 'CTTMeanReset'
    ]
    task_types = ['Uncoupled Baiting', 'Uncoupled Without Baiting', 'Coupled Baiting']
    later_stages = ['STAGE_FINAL', 'GRADUATED']
    
    # Create figure with subplots - now we need more space for 4 conditions
    fig, axes = plt.subplots(1, 3, figsize=(20, 8))
    
    # Colors for different data types
    colors = {
        'Data': '#2E86AB',
        'QLearning_L2F1_softmax': '#F18F01',
        'QLearning_L1F1_CK1_softmax': '#C73E1D',
        'ForagingCompareThreshold': '#A23B72',
        'CTTAvg': '#6A4C93',
        # 'CTTMeanReset': '#6A4C93'
    }
    
    # Bar width and positions for 4 conditions (rewarded run=1, rewarded run>1, unrewarded run=1, unrewarded run>1)
    bar_width = 0.12
    x_positions = np.arange(4)  # 4 conditions
    condition_labels = ['Rewarded\nRun=1', 'Rewarded\nRun>1', 'Unrewarded\nRun=1', 'Unrewarded\nRun>1']
    
    for task_idx, task_type in enumerate(task_types):
        ax = axes[task_idx]
        
        # Collect all results for this task type
        all_results = {}
        
        # Process real data
        print(f"Processing data for {task_type}...")
        df_task = df_session_history[
            (df_session_history['curriculum_name'] == task_type) &
            (df_session_history['current_stage_actual'].isin(later_stages))
        ]
        
        if len(df_task) > 0:
            conditional_data = compute_conditional_switch_probabilities(df_task, window_size)
            data_results = analyze_post_switch_probability_by_reward_and_run_length(conditional_data)
            all_results['Data'] = data_results
        
        # Process model results
        for model_type in model_types:
            key = (model_type, task_type)
            if key in dict_simulation_results:
                print(f"Processing {model_type} for {task_type}...")
                df_sim = dict_simulation_results[key].copy()
                df_sim = insert_switch_info(df_sim)
                conditional_sim = compute_conditional_switch_probabilities(df_sim, window_size)
                sim_results = analyze_post_switch_probability_by_reward_and_run_length(conditional_sim)
                all_results[model_type] = sim_results
        
        # Plot bars for each condition
        condition_mapping = [
            ('rewarded', 'run_length_1'),
            ('rewarded', 'run_length_gt1'),
            ('unrewarded', 'run_length_1'),
            ('unrewarded', 'run_length_gt1')
        ]
        
        for cond_idx, (reward_condition, run_condition) in enumerate(condition_mapping):
            x_pos = x_positions[cond_idx]
            
            for data_idx, (data_type, results) in enumerate(all_results.items()):
                if reward_condition in results and run_condition in results[reward_condition]:
                    condition_data = results[reward_condition][run_condition]
                    prob = condition_data['probability']
                    sem = condition_data['sem']
                    n = condition_data['n']
                    
                    # Position bars side by side
                    bar_pos = x_pos + (data_idx - 1.5) * bar_width
                    
                    # Plot bar
                    bar = ax.bar(bar_pos, prob, bar_width, 
                               yerr=sem, capsize=3,
                               color=colors[data_type], alpha=0.7,
                               label=data_type if cond_idx == 0 else "")
                    
                    # Add sample size annotation
                    if not np.isnan(prob):
                        ax.text(bar_pos, prob + sem + 0.01, f'n={n}',
                               ha='center', va='bottom', fontsize=7, rotation=90)
        
        # Customize subplot
        ax.set_xticks(x_positions)
        ax.set_xticklabels(condition_labels, fontsize=10)
        ax.set_ylabel('Switch probability at t+1')
        ax.set_title(f'{task_type}', fontsize=12)
        ax.set_ylim(0, None)
        ax.grid(True, alpha=0.3)
        
        # Add legend only to first subplot
        if task_idx == 0:
            ax.legend(bbox_to_anchor=(0, 1), loc='upper left', fontsize=10)
    
    plt.suptitle('Switch Probability at t+1: Data vs Models by Task Type, Reward and Run Length', 
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*100)
    print("SUMMARY STATISTICS")
    print("="*100)
    
    for task_type in task_types:
        print(f"\n{task_type}:")
        print("-" * len(task_type))
        
        # Real data
        df_task = df_session_history[
            (df_session_history['curriculum_name'] == task_type) &
            (df_session_history['current_stage_actual'].isin(later_stages))
        ]
        
        if len(df_task) > 0:
            conditional_data = compute_conditional_switch_probabilities(df_task, window_size)
            data_results = analyze_post_switch_probability_by_reward_and_run_length(conditional_data)
            
            print("Data:")
            for reward_condition in ['rewarded', 'unrewarded']:
                for run_condition in ['run_length_1', 'run_length_gt1']:
                    if reward_condition in data_results and run_condition in data_results[reward_condition]:
                        condition_data = data_results[reward_condition][run_condition]
                        prob = condition_data['probability']
                        sem = condition_data['sem']
                        n = condition_data['n']
                        run_label = 'Run=1' if run_condition == 'run_length_1' else 'Run>1'
                        print(f"  {reward_condition.capitalize()}, {run_label}: {prob:.3f} ± {sem:.3f} (n={n})")
        
        # Model results
        for model_type in model_types:
            key = (model_type, task_type)
            if key in dict_simulation_results:
                df_sim = dict_simulation_results[key].copy()
                df_sim = insert_switch_info(df_sim)
                conditional_sim = compute_conditional_switch_probabilities(df_sim, window_size)
                sim_results = analyze_post_switch_probability_by_reward_and_run_length(conditional_sim)
                
                print(f"{model_type}:")
                for reward_condition in ['rewarded', 'unrewarded']:
                    for run_condition in ['run_length_1', 'run_length_gt1']:
                        if reward_condition in sim_results and run_condition in sim_results[reward_condition]:
                            condition_data = sim_results[reward_condition][run_condition]
                            prob = condition_data['probability']
                            sem = condition_data['sem']
                            n = condition_data['n']
                            run_label = 'Run=1' if run_condition == 'run_length_1' else 'Run>1'
                            print(f"  {reward_condition.capitalize()}, {run_label}: {prob:.3f} ± {sem:.3f} (n={n})")

In [ ]:
# Usage
plot_pswitch_comparison(df_session_history, dict_simulation_results, window_size=10)

# Subject level analysis

In [ ]:
# without further conditioning on run length

# def compute_subject_level_conditional_pswitch(df_sessions, window_size=10):
#     """
#     Compute conditional p_switch at t+1 for individual subjects by aggregating across all their sessions.
    
#     Parameters:
#     -----------
#     df_sessions : pd.DataFrame
#         DataFrame with session data for multiple sessions per subject
#     window_size : int
#         Window size for switch analysis
        
#     Returns:
#     --------
#     subject_results : list of dicts
#         Each dict contains subject-level conditional p_switch results
#     """
#     subject_results = []
    
#     # Group by subject_id to aggregate all sessions per subject
#     for subject_id, subject_sessions in df_sessions.groupby('subject_id'):
#         # Compute conditional switch probabilities for all sessions of this subject
#         conditional_data = compute_conditional_switch_probabilities(subject_sessions, window_size)
        
#         # Analyze post-switch probability by reward (aggregated across all sessions)
#         results = analyze_post_switch_probability_by_reward(conditional_data)
        
#         subject_result = {
#             'subject_id': subject_id,
#             'n_sessions': len(subject_sessions),
#             'curriculum_name': subject_sessions['curriculum_name'].iloc[0],  # Assume same curriculum for subject
#             'rewarded_pswitch': results['rewarded']['probability'],
#             'rewarded_n': results['rewarded']['n'],
#             'unrewarded_pswitch': results['unrewarded']['probability'],
#             'unrewarded_n': results['unrewarded']['n']
#         }
#         subject_results.append(subject_result)
    
#     return subject_results


# def create_subject_level_comparison_dataframe(df_session_history, dict_simulation_results):
#     """
#     Create a comprehensive dataframe comparing subject-level conditional p_switch.
    
#     Parameters:
#     -----------
#     df_session_history : pd.DataFrame
#         Real behavioral data
#     dict_simulation_results : dict
#         Dictionary containing simulation results
        
#     Returns:
#     --------
#     df_comparison : pd.DataFrame
#         DataFrame with subject-level comparisons
#     """
#     model_types = ['ForagingCompareThreshold', 'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax']
#     task_types = ['Uncoupled Baiting', 'Uncoupled Without Baiting', 'Coupled Baiting']
#     later_stages = ['STAGE_FINAL', 'GRADUATED']
    
#     comparison_data = []
    
#     for task_type in task_types:
#         print(f"Processing {task_type}...")
        
#         # Get real data sessions for this task type
#         df_data_task = df_session_history[
#             (df_session_history['curriculum_name'] == task_type) &
#             (df_session_history['current_stage_actual'].isin(later_stages))
#         ]
        
#         if len(df_data_task) == 0:
#             print(f"  No real data found for {task_type}")
#             continue
        
#         # Compute subject-level p_switch for real data
#         data_subject_results = compute_subject_level_conditional_pswitch(df_data_task)
#         print(f"  Found {len(data_subject_results)} subjects in real data")
        
#         for model_type in model_types:
#             key = (model_type, task_type)
#             if key not in dict_simulation_results:
#                 continue
                
#             print(f"  Processing {model_type}...")
#             df_sim = dict_simulation_results[key].copy()
#             df_sim = insert_switch_info(df_sim)
            
#             # Compute subject-level p_switch for simulation data
#             sim_subject_results = compute_subject_level_conditional_pswitch(df_sim)
#             print(f"    Found {len(sim_subject_results)} subjects in simulation data")
            
#             # Match subjects between real data and simulation
#             for data_result in data_subject_results:
#                 subject_id = data_result['subject_id']
                
#                 # Find corresponding simulation result for same subject
#                 matching_sim = [s for s in sim_subject_results if s['subject_id'] == subject_id]
                
#                 if len(matching_sim) == 1:
#                     sim_result = matching_sim[0]
                    
#                     # Create comparison record
#                     comparison_record = {
#                         'subject_id': subject_id,
#                         'task_type': task_type,
#                         'model_type': model_type,
#                         'data_n_sessions': data_result['n_sessions'],
#                         'model_n_sessions': sim_result['n_sessions'],
#                         'data_rewarded_pswitch': data_result['rewarded_pswitch'],
#                         'data_rewarded_n': data_result['rewarded_n'],
#                         'data_unrewarded_pswitch': data_result['unrewarded_pswitch'],
#                         'data_unrewarded_n': data_result['unrewarded_n'],
#                         'model_rewarded_pswitch': sim_result['rewarded_pswitch'],
#                         'model_rewarded_n': sim_result['rewarded_n'],
#                         'model_unrewarded_pswitch': sim_result['unrewarded_pswitch'],
#                         'model_unrewarded_n': sim_result['unrewarded_n']
#                     }
#                     comparison_data.append(comparison_record)
    
#     df_comparison = pd.DataFrame(comparison_data)
#     print(f"\nTotal comparison records: {len(df_comparison)}")
    
#     return df_comparison


# def plot_subject_level_model_comparison(df_comparison):
#     """
#     Plot subject-level conditional p_switch comparison for each model, separated by task type.
    
#     Parameters:
#     -----------
#     df_comparison : pd.DataFrame
#         DataFrame with subject-level comparison data
#     """
#     from scipy.stats import pearsonr
#     import matplotlib.pyplot as plt
    
#     model_types = ['ForagingCompareThreshold', 'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax']
#     conditions = ['rewarded', 'unrewarded']
#     task_types = df_comparison['task_type'].unique()
    
#     # Create separate figure for each task type
#     for task_type in task_types:
#         print(f"\nPlotting {task_type}...")
#         df_task = df_comparison[df_comparison['task_type'] == task_type]
        
#         fig, axes = plt.subplots(3, 2, figsize=(12, 15))
        
#         for model_idx, model_type in enumerate(model_types):
#             df_model = df_task[df_task['model_type'] == model_type]
            
#             for cond_idx, condition in enumerate(conditions):
#                 ax = axes[model_idx, cond_idx]
                
#                 # Get data for this condition
#                 data_col = f'data_{condition}_pswitch'
#                 model_col = f'model_{condition}_pswitch'
#                 n_col = f'data_{condition}_n'
                
#                 # Filter out subjects with insufficient data (n < 5) or NaN values
#                 valid_mask = (
#                     (df_model[n_col] >= 5) & 
#                     (~df_model[data_col].isna()) & 
#                     (~df_model[model_col].isna())
#                 )
#                 df_valid = df_model[valid_mask]
                
#                 if len(df_valid) == 0:
#                     ax.text(0.5, 0.5, 'No valid data', ha='center', va='center', transform=ax.transAxes)
#                     ax.set_title(f'{model_type}\n{condition.capitalize()} (n=0)')
#                     continue
                
#                 x_data = df_valid[data_col].values
#                 y_data = df_valid[model_col].values
                
#                 # Create scatter plot with subject IDs as annotations
#                 scatter = ax.scatter(x_data, y_data, alpha=0.6, s=50)
                
#                 # Add diagonal line
#                 min_val = min(np.min(x_data), np.min(y_data))
#                 max_val = max(np.max(x_data), np.max(y_data))
#                 ax.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.7, linewidth=1)
                
#                 # Calculate metrics for deviation from diagonal line
#                 if len(x_data) > 1:
#                     # Mean Absolute Error from diagonal
#                     mae = np.mean(np.abs(y_data - x_data))
                    
#                     # Root Mean Square Error from diagonal
#                     rmse = np.sqrt(np.mean((y_data - x_data)**2))
                    
#                     # Mean bias (systematic over/under prediction)
#                     bias = np.mean(y_data - x_data)
                    
#                     # Add metrics text
#                     ax.text(0.05, 0.95, f'MAE = {mae:.3f}\nRMSE = {rmse:.3f}\nBias = {bias:.3f}\nn = {len(df_valid)}', 
#                            transform=ax.transAxes, verticalalignment='top',
#                            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                
#                 # Formatting
#                 ax.set_xlabel('Data p_switch')
#                 ax.set_ylabel('Model p_switch')
#                 ax.set_title(f'{model_type}\n{condition.capitalize()}')
#                 ax.grid(True, alpha=0.3)
                
#                 # Set equal aspect ratio
#                 ax.set_aspect('equal')
                
#                 # Set axis limits to show full range
#                 ax.set_xlim(min_val - 0.05, max_val + 0.05)
#                 ax.set_ylim(min_val - 0.05, max_val + 0.05)
        
#         plt.suptitle(f'Subject-level Conditional p_switch at t+1: Data vs Models\n{task_type}', 
#                      fontsize=14, y=0.98)
#         plt.tight_layout()
#         plt.show()


# def print_subject_level_summary(df_comparison):
#     """Print summary statistics for subject-level comparison, organized by task type."""
#     print("\n" + "="*80)
#     print("SUBJECT-LEVEL CONDITIONAL P_SWITCH COMPARISON SUMMARY")
#     print("="*80)
    
#     model_types = ['ForagingCompareThreshold', 'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax']
#     conditions = ['rewarded', 'unrewarded']
#     task_types = df_comparison['task_type'].unique()
    
#     for task_type in task_types:
#         print(f"\n{'='*60}")
#         print(f"TASK TYPE: {task_type}")
#         print(f"{'='*60}")
        
#         df_task = df_comparison[df_comparison['task_type'] == task_type]
        
#         for model_type in model_types:
#             print(f"\n{model_type}:")
#             print("-" * len(model_type))
            
#             df_model = df_task[df_task['model_type'] == model_type]
            
#             for condition in conditions:
#                 data_col = f'data_{condition}_pswitch'
#                 model_col = f'model_{condition}_pswitch'
#                 n_col = f'data_{condition}_n'
                
#                 # Filter valid data (subjects with sufficient switch events)
#                 valid_mask = (
#                     (df_model[n_col] >= 5) & 
#                     (~df_model[data_col].isna()) & 
#                     (~df_model[model_col].isna())
#                 )
#                 df_valid = df_model[valid_mask]
                
#                 if len(df_valid) > 0:
#                     x_data = df_valid[data_col].values
#                     y_data = df_valid[model_col].values
                    
#                     if len(x_data) > 1:
#                         # Calculate deviation metrics from diagonal line
#                         mae = np.mean(np.abs(y_data - x_data))
#                         rmse = np.sqrt(np.mean((y_data - x_data)**2))
#                         bias = np.mean(y_data - x_data)
                        
#                         print(f"  {condition.capitalize()}:")
#                         print(f"    Subjects: {len(df_valid)}")
#                         print(f"    MAE from diagonal: {mae:.3f}")
#                         print(f"    RMSE from diagonal: {rmse:.3f}")
#                         print(f"    Bias (model - data): {bias:.3f}")
#                         print(f"    Data mean: {np.mean(x_data):.3f} ± {np.std(x_data):.3f}")
#                         print(f"    Model mean: {np.mean(y_data):.3f} ± {np.std(y_data):.3f}")
                        
#                         # Additional subject-level statistics
#                         avg_sessions = np.mean(df_valid['data_n_sessions'])
#                         avg_switch_events = np.mean(df_valid[n_col])
#                         print(f"    Avg sessions per subject: {avg_sessions:.1f}")
#                         print(f"    Avg switch events per subject: {avg_switch_events:.1f}")
#                 else:
#                     print(f"  {condition.capitalize()}: No valid data")

In [ ]:
def compute_subject_level_conditional_pswitch(df_sessions, window_size=10):
    """
    Compute conditional p_switch at t+1 for individual subjects by aggregating across all their sessions.
    
    Parameters:
    -----------
    df_sessions : pd.DataFrame
        DataFrame with session data for multiple sessions per subject
    window_size : int
        Window size for switch analysis
        
    Returns:
    --------
    subject_results : list of dicts
        Each dict contains subject-level conditional p_switch results with run length conditioning
    """
    subject_results = []
    
    # Group by subject_id to aggregate all sessions per subject
    for subject_id, subject_sessions in df_sessions.groupby('subject_id'):
        # Compute conditional switch probabilities for all sessions of this subject
        conditional_data = compute_conditional_switch_probabilities(subject_sessions, window_size)
        
        # Analyze post-switch probability by reward and run length (aggregated across all sessions)
        results = analyze_post_switch_probability_by_reward_and_run_length(conditional_data)
        
        subject_result = {
            'subject_id': subject_id,
            'n_sessions': len(subject_sessions),
            'curriculum_name': subject_sessions['curriculum_name'].iloc[0],  # Assume same curriculum for subject
            # Rewarded conditions
            'rewarded_run1_pswitch': results['rewarded']['run_length_1']['probability'],
            'rewarded_run1_n': results['rewarded']['run_length_1']['n'],
            'rewarded_rungt1_pswitch': results['rewarded']['run_length_gt1']['probability'],
            'rewarded_rungt1_n': results['rewarded']['run_length_gt1']['n'],
            # Unrewarded conditions
            'unrewarded_run1_pswitch': results['unrewarded']['run_length_1']['probability'],
            'unrewarded_run1_n': results['unrewarded']['run_length_1']['n'],
            'unrewarded_rungt1_pswitch': results['unrewarded']['run_length_gt1']['probability'],
            'unrewarded_rungt1_n': results['unrewarded']['run_length_gt1']['n']
        }
        subject_results.append(subject_result)
    
    return subject_results


def create_subject_level_comparison_dataframe(df_session_history, dict_simulation_results):
    """
    Create a comprehensive dataframe comparing subject-level conditional p_switch with run length conditioning.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        Real behavioral data
    dict_simulation_results : dict
        Dictionary containing simulation results
        
    Returns:
    --------
    df_comparison : pd.DataFrame
        DataFrame with subject-level comparisons including run length conditions
    """
    model_types = ['ForagingCompareThreshold', 'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax']
    task_types = ['Uncoupled Baiting', 'Uncoupled Without Baiting', 'Coupled Baiting']
    later_stages = ['STAGE_FINAL', 'GRADUATED']
    
    comparison_data = []
    
    for task_type in task_types:
        print(f"Processing {task_type}...")
        
        # Get real data sessions for this task type
        df_data_task = df_session_history[
            (df_session_history['curriculum_name'] == task_type) &
            (df_session_history['current_stage_actual'].isin(later_stages))
        ]
        
        if len(df_data_task) == 0:
            print(f"  No real data found for {task_type}")
            continue
        
        # Compute subject-level p_switch for real data
        data_subject_results = compute_subject_level_conditional_pswitch(df_data_task)
        print(f"  Found {len(data_subject_results)} subjects in real data")
        
        for model_type in model_types:
            key = (model_type, task_type)
            if key not in dict_simulation_results:
                continue
                
            print(f"  Processing {model_type}...")
            df_sim = dict_simulation_results[key].copy()
            df_sim = insert_switch_info(df_sim)
            
            # Compute subject-level p_switch for simulation data
            sim_subject_results = compute_subject_level_conditional_pswitch(df_sim)
            print(f"    Found {len(sim_subject_results)} subjects in simulation data")
            
            # Match subjects between real data and simulation
            for data_result in data_subject_results:
                subject_id = data_result['subject_id']
                
                # Find corresponding simulation result for same subject
                matching_sim = [s for s in sim_subject_results if s['subject_id'] == subject_id]
                
                if len(matching_sim) == 1:
                    sim_result = matching_sim[0]
                    
                    # Create comparison record with all reward x run length combinations
                    comparison_record = {
                        'subject_id': subject_id,
                        'task_type': task_type,
                        'model_type': model_type,
                        'data_n_sessions': data_result['n_sessions'],
                        'model_n_sessions': sim_result['n_sessions'],
                        # Data - Rewarded conditions
                        'data_rewarded_run1_pswitch': data_result['rewarded_run1_pswitch'],
                        'data_rewarded_run1_n': data_result['rewarded_run1_n'],
                        'data_rewarded_rungt1_pswitch': data_result['rewarded_rungt1_pswitch'],
                        'data_rewarded_rungt1_n': data_result['rewarded_rungt1_n'],
                        # Data - Unrewarded conditions
                        'data_unrewarded_run1_pswitch': data_result['unrewarded_run1_pswitch'],
                        'data_unrewarded_run1_n': data_result['unrewarded_run1_n'],
                        'data_unrewarded_rungt1_pswitch': data_result['unrewarded_rungt1_pswitch'],
                        'data_unrewarded_rungt1_n': data_result['unrewarded_rungt1_n'],
                        # Model - Rewarded conditions
                        'model_rewarded_run1_pswitch': sim_result['rewarded_run1_pswitch'],
                        'model_rewarded_run1_n': sim_result['rewarded_run1_n'],
                        'model_rewarded_rungt1_pswitch': sim_result['rewarded_rungt1_pswitch'],
                        'model_rewarded_rungt1_n': sim_result['rewarded_rungt1_n'],
                        # Model - Unrewarded conditions
                        'model_unrewarded_run1_pswitch': sim_result['unrewarded_run1_pswitch'],
                        'model_unrewarded_run1_n': sim_result['unrewarded_run1_n'],
                        'model_unrewarded_rungt1_pswitch': sim_result['unrewarded_rungt1_pswitch'],
                        'model_unrewarded_rungt1_n': sim_result['unrewarded_rungt1_n']
                    }
                    comparison_data.append(comparison_record)
    
    df_comparison = pd.DataFrame(comparison_data)
    print(f"\nTotal comparison records: {len(df_comparison)}")
    
    return df_comparison


def plot_subject_level_model_comparison(df_comparison):
    """
    Plot subject-level conditional p_switch comparison for each model, separated by task type and including run length conditions.
    
    Parameters:
    -----------
    df_comparison : pd.DataFrame
        DataFrame with subject-level comparison data including run length conditions
    """
    from scipy.stats import pearsonr
    import matplotlib.pyplot as plt
    
    model_types = ['ForagingCompareThreshold', 'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax']
    # Now we have 4 conditions: rewarded_run1, rewarded_rungt1, unrewarded_run1, unrewarded_rungt1
    conditions = [
        ('rewarded_run1', 'Rewarded Run=1'),
        ('rewarded_rungt1', 'Rewarded Run>1'),
        ('unrewarded_run1', 'Unrewarded Run=1'),
        ('unrewarded_rungt1', 'Unrewarded Run>1')
    ]
    task_types = df_comparison['task_type'].unique()
    
    # Create separate figure for each task type
    for task_type in task_types:
        print(f"\nPlotting {task_type}...")
        df_task = df_comparison[df_comparison['task_type'] == task_type]
        
        fig, axes = plt.subplots(3, 4, figsize=(20, 15))  # 3 models x 4 conditions
        
        for model_idx, model_type in enumerate(model_types):
            df_model = df_task[df_task['model_type'] == model_type]
            
            for cond_idx, (condition, condition_label) in enumerate(conditions):
                ax = axes[model_idx, cond_idx]
                
                # Get data for this condition
                data_col = f'data_{condition}_pswitch'
                model_col = f'model_{condition}_pswitch'
                n_col = f'data_{condition}_n'
                
                # Filter out subjects with insufficient data (n < 5) or NaN values
                valid_mask = (
                    (df_model[n_col] >= 5) & 
                    (~df_model[data_col].isna()) & 
                    (~df_model[model_col].isna())
                )
                df_valid = df_model[valid_mask]
                
                if len(df_valid) == 0:
                    ax.text(0.5, 0.5, 'No valid data', ha='center', va='center', transform=ax.transAxes)
                    ax.set_title(f'{model_type}\n{condition_label} (n=0)', fontsize=10)
                    continue
                
                x_data = df_valid[data_col].values
                y_data = df_valid[model_col].values
                
                # Create scatter plot
                scatter = ax.scatter(x_data, y_data, alpha=0.6, s=40)
                
                # Add diagonal line
                min_val = min(np.min(x_data), np.min(y_data))
                max_val = max(np.max(x_data), np.max(y_data))
                ax.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.7, linewidth=1)
                
                # Calculate metrics for deviation from diagonal line
                if len(x_data) > 1:
                    # Mean Absolute Error from diagonal
                    mae = np.mean(np.abs(y_data - x_data))
                    
                    # Root Mean Square Error from diagonal
                    rmse = np.sqrt(np.mean((y_data - x_data)**2))
                    
                    # Mean bias (systematic over/under prediction)
                    bias = np.mean(y_data - x_data)
                    
                    # Add metrics text (smaller font for space)
                    ax.text(0.05, 0.95, f'MAE={mae:.2f}\nRMSE={rmse:.2f}\nBias={bias:.2f}\nn={len(df_valid)}', 
                           transform=ax.transAxes, verticalalignment='top', fontsize=8,
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                
                # Formatting
                ax.set_xlabel('Data p_switch', fontsize=9)
                ax.set_ylabel('Model p_switch', fontsize=9)
                ax.set_title(f'{model_type}\n{condition_label}', fontsize=10)
                ax.grid(True, alpha=0.3)
                ax.tick_params(labelsize=8)
                
                # Set equal aspect ratio
                ax.set_aspect('equal')
                
                # Set axis limits to show full range
                ax.set_xlim(min_val - 0.05, max_val + 0.05)
                ax.set_ylim(min_val - 0.05, max_val + 0.05)
        
        plt.suptitle(f'Subject-level Conditional p_switch at t+1: Data vs Models\n{task_type}', 
                     fontsize=16, y=0.98)
        plt.tight_layout()
        plt.show()


def print_subject_level_summary(df_comparison):
    """Print summary statistics for subject-level comparison with run length conditions, organized by task type."""
    print("\n" + "="*100)
    print("SUBJECT-LEVEL CONDITIONAL P_SWITCH COMPARISON SUMMARY (WITH RUN LENGTH)")
    print("="*100)
    
    model_types = ['ForagingCompareThreshold', 'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax']
    conditions = [
        ('rewarded_run1', 'Rewarded Run=1'),
        ('rewarded_rungt1', 'Rewarded Run>1'),
        ('unrewarded_run1', 'Unrewarded Run=1'),
        ('unrewarded_rungt1', 'Unrewarded Run>1')
    ]
    task_types = df_comparison['task_type'].unique()
    
    for task_type in task_types:
        print(f"\n{'='*80}")
        print(f"TASK TYPE: {task_type}")
        print(f"{'='*80}")
        
        df_task = df_comparison[df_comparison['task_type'] == task_type]
        
        for model_type in model_types:
            print(f"\n{model_type}:")
            print("-" * len(model_type))
            
            df_model = df_task[df_task['model_type'] == model_type]
            
            for condition, condition_label in conditions:
                data_col = f'data_{condition}_pswitch'
                model_col = f'model_{condition}_pswitch'
                n_col = f'data_{condition}_n'
                
                # Filter valid data (subjects with sufficient switch events)
                valid_mask = (
                    (df_model[n_col] >= 5) & 
                    (~df_model[data_col].isna()) & 
                    (~df_model[model_col].isna())
                )
                df_valid = df_model[valid_mask]
                
                if len(df_valid) > 0:
                    x_data = df_valid[data_col].values
                    y_data = df_valid[model_col].values
                    
                    if len(x_data) > 1:
                        # Calculate deviation metrics from diagonal line
                        mae = np.mean(np.abs(y_data - x_data))
                        rmse = np.sqrt(np.mean((y_data - x_data)**2))
                        bias = np.mean(y_data - x_data)
                        
                        print(f"  {condition_label}:")
                        print(f"    Subjects: {len(df_valid)}")
                        print(f"    MAE from diagonal: {mae:.3f}")
                        print(f"    RMSE from diagonal: {rmse:.3f}")
                        print(f"    Bias (model - data): {bias:.3f}")
                        print(f"    Data mean: {np.mean(x_data):.3f} ± {np.std(x_data):.3f}")
                        print(f"    Model mean: {np.mean(y_data):.3f} ± {np.std(y_data):.3f}")
                        
                        # Additional subject-level statistics
                        avg_sessions = np.mean(df_valid['data_n_sessions'])
                        avg_switch_events = np.mean(df_valid[n_col])
                        print(f"    Avg sessions per subject: {avg_sessions:.1f}")
                        print(f"    Avg switch events per subject: {avg_switch_events:.1f}")
                else:
                    print(f"  {condition_label}: No valid data")

In [ ]:
# Create subject-level comparison dataframe
print("Creating subject-level comparison dataframe...")
df_subject_comparison = create_subject_level_comparison_dataframe(df_session_history, dict_simulation_results)

# # Save the comparison dataframe
# comparison_file = os.path.expanduser("~/capsule/analysis_result/df_subject_level_pswitch_comparison.pkl")
# df_subject_comparison.to_pickle(comparison_file)
# print(f"Saved subject-level comparison to {comparison_file}")

# Display sample of the comparison data
print("\nSample of comparison data:")
print(df_subject_comparison.head())
print(f"\nDataFrame shape: {df_subject_comparison.shape}")
print(f"Columns: {list(df_subject_comparison.columns)}")

# Plot the comparison
plot_subject_level_model_comparison(df_subject_comparison)

# Print summary statistics
print_subject_level_summary(df_subject_comparison)

# Session level analysis

In [ ]:
def compute_session_level_conditional_pswitch(df_sessions, window_size=10):
    """
    Compute conditional p_switch at t+1 for individual sessions.
    
    Parameters:
    -----------
    df_sessions : pd.DataFrame
        DataFrame with session data
    window_size : int
        Window size for switch analysis
        
    Returns:
    --------
    session_results : list of dicts
        Each dict contains session-level conditional p_switch results
    """
    session_results = []
    
    for _, row in df_sessions.iterrows():
        session_id = (row['subject_id'], row['session_date'])
        
        # Create single-session dataframe
        df_single_session = pd.DataFrame([row])
        
        # Compute conditional switch probabilities for this session
        conditional_data = compute_conditional_switch_probabilities(df_single_session, window_size)
        
        # Analyze post-switch probability by reward
        results = analyze_post_switch_probability_by_reward(conditional_data)
        
        session_result = {
            'subject_id': row['subject_id'],
            'session_date': row['session_date'],
            'curriculum_name': row.get('curriculum_name', None),
            'rewarded_pswitch': results['rewarded']['probability'],
            'rewarded_n': results['rewarded']['n'],
            'unrewarded_pswitch': results['unrewarded']['probability'],
            'unrewarded_n': results['unrewarded']['n']
        }
        session_results.append(session_result)
    
    return session_results


def create_session_level_comparison_dataframe(df_session_history, dict_simulation_results):
    """
    Create a comprehensive dataframe comparing session-level conditional p_switch.
    
    Parameters:
    -----------
    df_session_history : pd.DataFrame
        Real behavioral data
    dict_simulation_results : dict
        Dictionary containing simulation results
        
    Returns:
    --------
    df_comparison : pd.DataFrame
        DataFrame with session-level comparisons
    """
    model_types = ['ForagingCompareThreshold', 'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax']
    task_types = ['Uncoupled Baiting', 'Uncoupled Without Baiting', 'Coupled Baiting']
    later_stages = ['STAGE_FINAL', 'GRADUATED']
    
    comparison_data = []
    
    for task_type in task_types:
        print(f"Processing {task_type}...")
        
        # Get real data sessions for this task type
        df_data_task = df_session_history[
            (df_session_history['curriculum_name'] == task_type) &
            (df_session_history['current_stage_actual'].isin(later_stages))
        ]
        
        for model_type in model_types:
            key = (model_type, task_type)
            if key not in dict_simulation_results:
                continue
                
            print(f"  Processing {model_type}...")
            df_sim = dict_simulation_results[key].copy()
            df_sim = insert_switch_info(df_sim)
            
            # Get unique sessions from simulation data
            sim_sessions = df_sim.groupby(['subject_id', 'session_date']).first().reset_index()
            print(f"    Found {len(sim_sessions)} unique simulated sessions")
            
            # For each unique session, compute model p_switch
            sim_results = compute_session_level_conditional_pswitch(df_sim)
            
            # Match with real data sessions
            for sim_result in sim_results:
                subject_id = sim_result['subject_id']
                session_date = sim_result['session_date']
                
                # Find corresponding real data session
                matching_data = df_data_task[
                    (df_data_task['subject_id'] == subject_id) &
                    (df_data_task['session_date'] == session_date)
                ]
                
                if len(matching_data) == 1:
                    # Compute real data p_switch for this session
                    data_results = compute_session_level_conditional_pswitch(matching_data)
                    
                    if len(data_results) == 1:
                        data_result = data_results[0]
                        
                        # Create comparison record
                        comparison_record = {
                            'subject_id': subject_id,
                            'session_date': session_date,
                            'task_type': task_type,
                            'model_type': model_type,
                            'data_rewarded_pswitch': data_result['rewarded_pswitch'],
                            'data_rewarded_n': data_result['rewarded_n'],
                            'data_unrewarded_pswitch': data_result['unrewarded_pswitch'],
                            'data_unrewarded_n': data_result['unrewarded_n'],
                            'model_rewarded_pswitch': sim_result['rewarded_pswitch'],
                            'model_rewarded_n': sim_result['rewarded_n'],
                            'model_unrewarded_pswitch': sim_result['unrewarded_pswitch'],
                            'model_unrewarded_n': sim_result['unrewarded_n']
                        }
                        comparison_data.append(comparison_record)
    
    df_comparison = pd.DataFrame(comparison_data)
    print(f"\nTotal comparison records: {len(df_comparison)}")
    
    return df_comparison


def plot_session_level_model_comparison(df_comparison):
    """
    Plot session-level conditional p_switch comparison for each model, separated by task type.
    
    Parameters:
    -----------
    df_comparison : pd.DataFrame
        DataFrame with session-level comparison data
    """
    from scipy.stats import pearsonr
    import matplotlib.pyplot as plt
    
    model_types = ['ForagingCompareThreshold', 'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax']
    conditions = ['rewarded', 'unrewarded']
    task_types = df_comparison['task_type'].unique()
    
    # Create separate figure for each task type
    for task_type in task_types:
        print(f"\nPlotting {task_type}...")
        df_task = df_comparison[df_comparison['task_type'] == task_type]
        
        fig, axes = plt.subplots(3, 2, figsize=(12, 15))
        
        for model_idx, model_type in enumerate(model_types):
            df_model = df_task[df_task['model_type'] == model_type]
            
            for cond_idx, condition in enumerate(conditions):
                ax = axes[model_idx, cond_idx]
                
                # Get data for this condition
                data_col = f'data_{condition}_pswitch'
                model_col = f'model_{condition}_pswitch'
                n_col = f'data_{condition}_n'
                
                # Filter out sessions with insufficient data (n < 3) or NaN values
                valid_mask = (
                    (df_model[n_col] >= 3) & 
                    (~df_model[data_col].isna()) & 
                    (~df_model[model_col].isna())
                )
                df_valid = df_model[valid_mask]
                
                if len(df_valid) == 0:
                    ax.text(0.5, 0.5, 'No valid data', ha='center', va='center', transform=ax.transAxes)
                    ax.set_title(f'{model_type}\n{condition.capitalize()} (n=0)')
                    continue
                
                x_data = df_valid[data_col].values
                y_data = df_valid[model_col].values
                
                # Create scatter plot
                scatter = ax.scatter(x_data, y_data, alpha=0.6, s=30)
                
                # Add diagonal line
                min_val = min(np.min(x_data), np.min(y_data))
                max_val = max(np.max(x_data), np.max(y_data))
                ax.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.7, linewidth=1)
                
                # Calculate metrics for deviation from diagonal line
                if len(x_data) > 1:
                    # Mean Absolute Error from diagonal
                    mae = np.mean(np.abs(y_data - x_data))
                    
                    # Root Mean Square Error from diagonal
                    rmse = np.sqrt(np.mean((y_data - x_data)**2))
                    
                    # Mean bias (systematic over/under prediction)
                    bias = np.mean(y_data - x_data)
                    
                    # Add metrics text
                    ax.text(0.05, 0.95, f'MAE = {mae:.3f}\nRMSE = {rmse:.3f}\nBias = {bias:.3f}\nn = {len(df_valid)}', 
                           transform=ax.transAxes, verticalalignment='top',
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                
                # Formatting
                ax.set_xlabel('Data p_switch')
                ax.set_ylabel('Model p_switch')
                ax.set_title(f'{model_type}\n{condition.capitalize()}')
                ax.grid(True, alpha=0.3)
                
                # Set equal aspect ratio
                ax.set_aspect('equal')
                
                # Set axis limits to show full range
                ax.set_xlim(min_val - 0.05, max_val + 0.05)
                ax.set_ylim(min_val - 0.05, max_val + 0.05)
        
        plt.suptitle(f'Session-level Conditional p_switch at t+1: Data vs Models\n{task_type}', 
                     fontsize=14, y=0.98)
        plt.tight_layout()
        plt.show()


def print_session_level_summary(df_comparison):
    """Print summary statistics for session-level comparison, organized by task type."""
    print("\n" + "="*80)
    print("SESSION-LEVEL CONDITIONAL P_SWITCH COMPARISON SUMMARY")
    print("="*80)
    
    model_types = ['ForagingCompareThreshold', 'QLearning_L2F1_softmax', 'QLearning_L1F1_CK1_softmax']
    conditions = ['rewarded', 'unrewarded']
    task_types = df_comparison['task_type'].unique()
    
    for task_type in task_types:
        print(f"\n{'='*60}")
        print(f"TASK TYPE: {task_type}")
        print(f"{'='*60}")
        
        df_task = df_comparison[df_comparison['task_type'] == task_type]
        
        for model_type in model_types:
            print(f"\n{model_type}:")
            print("-" * len(model_type))
            
            df_model = df_task[df_task['model_type'] == model_type]
            
            for condition in conditions:
                data_col = f'data_{condition}_pswitch'
                model_col = f'model_{condition}_pswitch'
                n_col = f'data_{condition}_n'
                
                # Filter valid data
                valid_mask = (
                    (df_model[n_col] >= 3) & 
                    (~df_model[data_col].isna()) & 
                    (~df_model[model_col].isna())
                )
                df_valid = df_model[valid_mask]
                
                if len(df_valid) > 0:
                    x_data = df_valid[data_col].values
                    y_data = df_valid[model_col].values
                    
                    if len(x_data) > 1:
                        # Calculate deviation metrics from diagonal line
                        mae = np.mean(np.abs(y_data - x_data))
                        rmse = np.sqrt(np.mean((y_data - x_data)**2))
                        bias = np.mean(y_data - x_data)
                        
                        print(f"  {condition.capitalize()}:")
                        print(f"    Sessions: {len(df_valid)}")
                        print(f"    MAE from diagonal: {mae:.3f}")
                        print(f"    RMSE from diagonal: {rmse:.3f}")
                        print(f"    Bias (model - data): {bias:.3f}")
                        print(f"    Data mean: {np.mean(x_data):.3f} ± {np.std(x_data):.3f}")
                        print(f"    Model mean: {np.mean(y_data):.3f} ± {np.std(y_data):.3f}")
                else:
                    print(f"  {condition.capitalize()}: No valid data")

In [ ]:
# Create session-level comparison dataframe
print("Creating session-level comparison dataframe...")
df_session_comparison = create_session_level_comparison_dataframe(df_session_history, dict_simulation_results)

# # Save the comparison dataframe
# comparison_file = os.path.expanduser("~/capsule/analysis_result/df_session_level_pswitch_comparison.pkl")
# df_session_comparison.to_pickle(comparison_file)
# print(f"Saved session-level comparison to {comparison_file}")

# Display sample of the comparison data
print("\nSample of comparison data:")
print(df_session_comparison.head())

# Plot the comparison
plot_session_level_model_comparison(df_session_comparison)

# Print summary statistics
print_session_level_summary(df_session_comparison)